<a href="https://colab.research.google.com/github/leonc10/CreativeHub-DataEngineeringModule1-Classwork/blob/main/notebooks/session02_python_functions_files.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 2 — Python: decisions, loops, functions and files

Last session we stored data in lists and dicts. Today we make Python **do work** on that data:
make decisions, repeat steps, package logic into functions, deal with errors and read real files.

By the end you will have cleaned Wavely's messy listening log **using only plain Python** —
and you will see why data engineers need a tool like Spark.

## Homework review (15 min)
Open `homework/hw01_python_basics.ipynb`. We go through it together.

---
Run the setup cell below. It finds the course data folder (or downloads it in Colab). You don't need to understand it yet.

In [1]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

Course data: /content/course/data


---
## Part 1 — Decisions with `if`

Comparisons give a `bool`: `==` equal, `!=` not equal, `<`, `<=`, `>`, `>=`.
Combine them with `and`, `or`, `not`.

Wavely's rule: **a play shorter than 30 seconds is a skip** and does not count for artist payments.

In [2]:
seconds = 12

if seconds < 30:
    print("skip")
elif seconds < 120:
    print("short listen")
else:
    print("proper listen")

skip


The indented lines (4 spaces) belong to the `if`. Python checks the conditions from top to bottom and runs **only the first** matching block.

### Exercise 1 (Core) — Classify a play
Set `label` to:
- `"skip"` if `seconds` is under 30
- `"full"` if `seconds` is at least 95% of `duration`
- `"partial"` otherwise

Try changing `seconds` to `10` and `200` to test all branches.

In [12]:
seconds = 150
duration = 203

# YOUR CODE HERE
if seconds < 30:
  label = 'skip'
elif seconds >= 0.95*duration:
  label = 'full'
else:
  label = 'partial'


print(label)  # expected for 150/203: partial

partial


---
## Part 2 — Loops with `for`

A `for` loop runs the indented block once for every item in a list.

In [13]:
plays = [
    {"artist": "Dua Lipa", "seconds": 226},
    {"artist": "Drake", "seconds": 6},
    {"artist": "Coldplay", "seconds": 280},
    {"artist": "Dua Lipa", "seconds": 190},
    {"artist": "Drake", "seconds": 241},
]

for play in plays:
    print(play["artist"], "->", play["seconds"])

Dua Lipa -> 226
Drake -> 6
Coldplay -> 280
Dua Lipa -> 190
Drake -> 241


Two patterns you will use **all the time**:

1. **Accumulator** — start at zero, add in each round.
2. **Counting with a dict** — `counts.get(key, 0) + 1` means "the current count, or 0 if we haven't seen this key yet, plus one".

In [14]:
total = 0
for play in plays:
    total = total + play["seconds"]
print("total seconds:", total)

plays_per_artist = {}
for play in plays:
    artist = play["artist"]
    plays_per_artist[artist] = plays_per_artist.get(artist, 0) + 1
print(plays_per_artist)

total seconds: 943
{'Dua Lipa': 2, 'Drake': 2, 'Coldplay': 1}


### Exercise 2 (Core) — Loop over plays
Using `plays` from above:
1. `skips` — number of plays under 30 seconds
2. `seconds_per_artist` — a dict: artist → total seconds listened, **not counting skips**

In [51]:
# YOUR CODE HERE
skips = 0

seconds_per_artist = {}
for play in plays:
  if play['seconds']< 30:
    skips =+ 1

  else:
    artist = play['artist']
    seconds_per_artist[artist] = seconds_per_artist.get(artist, 0) + play['seconds']
print(skips, seconds_per_artist)  # expected: 1 {'Dua Lipa': 416, 'Coldplay': 280, 'Drake': 241}

1 {'Dua Lipa': 416, 'Coldplay': 280, 'Drake': 241}


**Stretch:** do the same with a list comprehension, e.g. `[p for p in plays if p["seconds"] >= 30]`, then loop over the result.

---
## Part 3 — Functions

A **function** packages steps under a name so you can reuse (and test) them.
`def` defines it, parameters go in `( )`, and `return` sends the result back.

Data engineers love functions: the same cleaning rule must be applied *identically* to every row, every day.

In [52]:
def is_skip(seconds, threshold=30):
    """True if a play is too short to count."""
    return seconds < threshold


print(is_skip(12))
print(is_skip(200))
print(is_skip(40, threshold=60))  # parameters can have default values

True
False
True


### Exercise 3 (Core) — Two small helpers
1. `ms_to_text(ms)` returns `"m:ss"` for a number of milliseconds (you did this in homework 1).
2. `clean_name(text)` strips spaces and fixes capitalisation, e.g. `"  dua LIPA "` → `"Dua Lipa"`.

In [59]:
def ms_to_text(ms):
    # YOUR CODE HERE
    seconds = ms // 1000
    return f"{seconds//60}:{seconds%60:02d}"

def clean_name(text):
    # YOUR CODE HERE
    return text.strip().title()

print(ms_to_text(185000), ms_to_text(61000))  # expected: 3:05 1:01
print(clean_name("  dua LIPA "))  # expected: Dua Lipa

3:05 1:01
Dua Lipa


---
## Part 4 — When things go wrong: `try` / `except`

Converting text to a number crashes if the text is not a number:

In [61]:
print(int("225"))
print(int("unknown"))  # remove the # at the start and run: ValueError!

225


ValueError: invalid literal for int() with base 10: 'unknown'

In data pipelines, one bad value must **not** stop the processing of millions of good ones.
`try` / `except` lets us catch the error and decide what to do instead (for example: use `None`).

In [62]:
def to_int(text):
    try:
        return int(text)
    except ValueError:
        return None


print(to_int("225"), to_int("unknown"), to_int(""))

225 None None


### Exercise 4 (Core) — `parse_duration`
In the messy log, the `duration` column looks like this: `"3:45"`, `"225"`, `"225s"`, `""` or `"unknown"`.
Write `parse_duration(text)` that returns the number of **seconds as an int**, or `None` if it cannot be parsed.

The `assert` lines test your function: if one fails you get an `AssertionError`.

In [87]:
def parse_duration(text):
    # YOUR CODE HERE

    try:
      if ':' in text:
        minutes, seconds = text.split(':')
        return int(minutes)*60+int(seconds)

      if text.endswith('s'):
        return int(text[:-1])
      if text == "":
        return None
      if text == "unknown":
        return None
      return int(text)
    except AssertionError:
      return None



assert parse_duration("3:45") == 225
assert parse_duration("225") == 225
assert parse_duration("225s") == 225
assert parse_duration("") is None
assert parse_duration("unknown") is None
print("All tests passed!")

All tests passed!


> **Under the hood:** what you just wrote is a **unit test**. Professional pipelines have hundreds of them, and they run automatically before any code change goes live.

---
## Part 5 — Reading a real file

Files live in folders. `pathlib.Path` builds paths that work on Windows, Mac and Linux. `DATA` (from the setup cell) points to the course data folder.

A **CSV** file is a text table: one row per line, values separated by commas. Python's `csv.DictReader` gives us each row as a dict — the list-of-dicts table from session 1!

In [88]:
import csv

log_path = DATA / "messy" / "listening_log_messy.csv"

with open(log_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(len(rows), "rows")
print(rows[0])
print(rows[6])

625 rows
{'play_id': '1', 'played_at': '01/09/2026 23:56', 'user': 'U0293', 'artist': 'Calvin Harris', 'track': ' Little Promises  ', 'duration': '47', 'country': 'DE', 'device': 'mobile'}
{'play_id': '7', 'played_at': '1 Sep 2026, 04:31', 'user': 'U0676', 'artist': 'Bad Bunny', 'track': 'Endless Thunder', 'duration': '1:46', 'country': 'AT', 'device': 'SMART_SPEAKER'}


`with open(...) as f:` opens the file and closes it automatically afterwards.

Let's look at the problems. A **set** keeps only distinct values — great for spotting variants:

In [89]:
devices = set()
for row in rows:
    devices.add(row["device"])
print(devices)

print({row["country"] for row in rows})  # the same idea in one line (a set comprehension)

{'SMART_SPEAKER', 'mobile', 'MOBILE', 'Mobile', 'smart_speaker', 'WEB', 'Smart_speaker', 'desktop', 'Desktop', 'web', 'Web', 'smart speaker', 'DESKTOP'}
{'', ' AT ', ' HU ', 'DE', ' CZ ', 'SK', 'HU', 'sk', 'at', 'PL', 'pl', ' DE ', ' PL ', 'AT', 'cz', 'hu', 'CZ', 'de', ' SK '}


In [95]:
print({row["played_at"] for row in rows})
print({row["artist"] for row in rows})
print({row["track"] for row in rows})
print({row["duration"] for row in rows})
print({row["device"] for row in rows})

{'07/09/2026 18:48', '2026-09-07 07:16:11', '1 Sep 2026, 07:37', '2026-09-07 06:05:56', '06/09/2026 00:15', '2026-09-01 16:58:33', '3 Sep 2026, 23:33', '2026-09-05 08:53:12', '2026-09-02 19:50:35', '2026-09-04 19:20:19', '3 Sep 2026, 16:36', '2026-09-07 20:43:48', '05/09/2026 10:11', '2 Sep 2026, 21:08', '2026-09-05 08:41:50', '2026-09-07 15:16:26', '2026-09-07 12:36:56', '7 Sep 2026, 08:30', '2026-09-02 23:11:47', '2026-09-03 09:49:40', '2026-09-04 07:33:45', '07/09/2026 17:38', '2026-09-04 12:03:39', '2026-09-06 06:03:10', '4 Sep 2026, 16:27', '02/09/2026 08:20', '6 Sep 2026, 07:54', '01/09/2026 18:56', '04/09/2026 16:37', '2026-09-03 23:07:05', '2026-09-06 20:19:14', '4 Sep 2026, 12:00', '4 Sep 2026, 07:52', '2026-09-02 06:35:56', '06/09/2026 08:01', '2026-09-05 14:09:31', '2026-09-07 20:17:30', '3 Sep 2026, 12:42', '3 Sep 2026, 22:21', '2026-09-04 19:36:05', '2026-09-07 21:08:57', '2026-09-06 19:42:55', '05/09/2026 07:33', '4 Sep 2026, 12:11', '2026-09-03 21:16:09', '2026-09-05 18:

**Discuss:** which problems do you see? Look at `played_at`, `artist`, `track`, `duration`, `country`, `device`.
Are there duplicate rows?

---
## Part 6 — A mini pipeline in plain Python

Dates come in three formats. `datetime.strptime(text, format)` parses a date if it matches the format, and raises `ValueError` if not.
Codes: `%Y` year, `%m` month, `%d` day, `%H` hour, `%M` minute, `%S` second, `%b` short month name (`Sep`).

In [ ]:
from datetime import datetime

print(datetime.strptime("2026-09-03 18:22:05", "%Y-%m-%d %H:%M:%S"))
print(datetime.strptime("03/09/2026 18:22", "%d/%m/%Y %H:%M"))

### Exercise 5 (Core) — `parse_timestamp`
Try each format in `FORMATS` in turn; return the first one that works. Return `None` if none works.

In [ ]:
FORMATS = ["%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M", "%d %b %Y, %H:%M"]


def parse_timestamp(text):
    # YOUR CODE HERE
    ...


assert parse_timestamp("2026-09-03 18:22:05") == datetime(2026, 9, 3, 18, 22, 5)
assert parse_timestamp("03/09/2026 18:22") == datetime(2026, 9, 3, 18, 22)
assert parse_timestamp("1 Sep 2026, 04:31") == datetime(2026, 9, 1, 4, 31)
assert parse_timestamp("yesterday") is None
print("All tests passed!")

### Exercise 6 (Core) — Clean the whole log
Complete `clean_row(row)` so it returns a **new** dict with:

| key | rule |
|---|---|
| `play_id` | `int` |
| `played_at` | `parse_timestamp(...)` |
| `user` | unchanged |
| `artist`, `track` | `clean_name(...)` |
| `seconds` | `parse_duration(...)` of the `duration` column |
| `country` | stripped, upper-case; `None` if empty |
| `device` | stripped, lower-case, spaces replaced by `_` |

The loop below then removes duplicate rows and keeps the clean ones.

In [ ]:
def clean_row(row):
    # YOUR CODE HERE
    ...


clean_rows = []
seen_ids = set()
for row in rows:
    if row["play_id"] in seen_ids:
        continue  # duplicate: skip to the next row
    seen_ids.add(row["play_id"])
    clean_rows.append(clean_row(row))

print(len(rows), "->", len(clean_rows), "rows")  # expected: 625 -> 600 rows
print(clean_rows[6])
print({r["device"] for r in clean_rows})

### Exercise 7 (Core) — Answer a question
**Which 5 artists got the most listening minutes?** Ignore rows where `seconds` is `None`.
Build a dict artist → minutes, then sort it.

`sorted(d.items(), key=lambda item: item[1], reverse=True)` sorts `(key, value)` pairs by value, largest first.

In [ ]:
# YOUR CODE HERE

for artist, minutes in top5:
    print(f"{artist:<15} {minutes:6.1f} min")

### Exercise 8 (Stretch) — Save the clean data
Write `clean_rows` to `output/listening_log_clean.csv` with `csv.DictWriter`. Read it back to check.

In [ ]:
OUTPUT = DATA.parent / "output"
OUTPUT.mkdir(exist_ok=True)

# YOUR CODE HERE

---
## Why we need something bigger

Let's time our pipeline and do some quick maths.

In [ ]:
import time

start = time.perf_counter()
for _ in range(20):  # repeat so the timing is measurable
    cleaned = [clean_row(r) for r in rows]
seconds_per_row = (time.perf_counter() - start) / (20 * len(rows))

rows_per_day = 1_000_000_000  # a large streaming service
hours = seconds_per_row * rows_per_day / 3600
print(f"{seconds_per_row * 1e6:.1f} microseconds per row -> {hours:.1f} hours per day of data on one CPU core")

And that is only the time — a billion rows also don't fit in a laptop's memory.

**The idea behind Spark:** split the data into many pieces (**partitions**) and let many CPU cores
on many machines clean them **at the same time**. Same logic, much more power. That starts next session.

## Wrap-up

| Concept | Example |
|---|---|
| decisions | `if` / `elif` / `else`, `==`, `<`, `and`, `or` |
| loops | `for row in rows:`, accumulator, counting dict |
| functions | `def parse_duration(text): ... return ...` |
| errors | `try:` / `except ValueError:` |
| files | `Path`, `open`, `csv.DictReader`, `csv.DictWriter` |
| testing | `assert f(x) == expected` |

**Homework:** `homework/hw02_python_files.ipynb`